In [1]:
from pathlib import Path
from typing import List

import numpy as np
import tifffile
from bioio import BioImage
from cellpose import models

import skimage

import napari



Welcome to CellposeSAM, cellpose v
cellpose version: 	4.0.4 
platform:       	win32 
python version: 	3.10.18 
torch version:  	2.7.1+cu126! The neural network component of
CPSAM is much larger than in previous versions and CPU excution is slow. 
We encourage users to use GPU/MPS if available. 




In [11]:
pip install napari

  Using cached napari-0.7.0-py3-none-any.whl.metadata (17 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached app_model-0.5.1-py3-none-any.whl.metadata (3.2 kB)
  Using cached cachey-0.2.1-py3-none-any.whl.metadata (2.2 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached magicgui-0.10.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached napari_console-0.1.4-py3-none-any.whl.metadata (5.7 kB)
  Using cached napari_plugin_engine-0.2.1-py3-none-any.whl.metadata (4.5 kB)
  Using cached napari_svg-0.2.1-py3-none-any.whl.metadata (3.6 kB)
  Using cached npe2-0.8.2-py3-none-any.whl.metadata (6.8 kB)
  Using cached psygnal-0.15.1-cp310-cp310-win_amd64.whl.metadata (6.1 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 kB)
  Using cached pyopengl-3.1.10-py3-none-any.whl.metadata (3.3 kB)
  Using cached superqt-0.8.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached in_n_out-0.2.1-py3-none-any.whl.metad

In [2]:
filename = r"D:\OneDrive - IEM\000_inbox\260604_Kralova_python_troubleshooting\cellpose + far-red channel\motor cortex\1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-22_14.10.39_FusionStitcher.ims"

image = BioImage(str(filename))
image.scenes[2]

'1.3_P301S+PLA_MC_AT8_E2_first bottom_2026-04-22_14.10.39_FusionStitcher.ims Resolution Level 3'

In [6]:
image.shape

(1, 4, 13, 9114, 9338)

In [7]:
image.set_scene(image.scenes[2])
image.shape

(1, 4, 13, 2278, 2334)

In [16]:
image.set_scene(image.scenes[2])

arr = image.get_image_data()
arr = arr.squeeze()
arr.shape

(4, 13, 2278, 2334)

In [17]:
viewer = napari.Viewer()

In [18]:
viewer.add_image(arr, name="raw_downscaled_4")

<Image layer 'raw_downscaled_4' at 0x179d137a1d0>

In [26]:
ROI = viewer.layers["crop_ROI"].data

print(ROI)

x_min = round(min(ROI[0][0][2], ROI[0][2][2]))
y_min = round(min(ROI[0][0][3], ROI[0][2][3]))
x_max = round(max(ROI[0][0][2], ROI[0][2][2]))
y_max = round(max(ROI[0][0][3], ROI[0][2][3]))

print(x_min, y_min, x_max, y_max)

green_crop = arr[1, :, x_min:x_max, y_min:y_max]
green_crop.shape

[array([[1.0000000e+00, 6.0000000e+00, 3.3632660e+02, 4.6136371e+02],
       [1.0000000e+00, 6.0000000e+00, 3.3632660e+02, 1.0220226e+03],
       [1.0000000e+00, 6.0000000e+00, 9.0992340e+02, 1.0220226e+03],
       [1.0000000e+00, 6.0000000e+00, 9.0992340e+02, 4.6136371e+02]],
      dtype=float32)]
336 461 910 1022


(13, 574, 561)

In [ ]:
viewer.add_image(green_crop, name="green_crop")

<Image layer 'green_crop' at 0x179d18d8f10>

In [ ]:
# anisotropy factor
1.82 / (0.152 * 4)

2.993421052631579

In [29]:
# Cellpose settings
CELLPROB_THRESHOLD = -5.0
USE_GPU = True
MODEL_TYPE = "sam"

model = models.CellposeModel(gpu=USE_GPU, model_type=MODEL_TYPE)

masks_zyx, _flows, _styles = model.eval(
            green_crop,
            channels=[0, 0],
            do_3D=True,
            z_axis=0,
            cellprob_threshold=CELLPROB_THRESHOLD
        )

viewer.add_labels(masks_zyx, name="masks_zyx")

model_type argument is not used in v4.0.1+. Ignoring this argument...
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used


<Labels layer 'masks_zyx' at 0x179d13a76a0>

In [ ]:
background = skimage.restoration.rolling_ball(green_crop, radius=5)